# Build h3_density (Legacy)
Runs `build_h3_density()` standalone and outputs `h3_density.csv`.
Use this to inspect/test the aggregation without running the full ETL.

Run from the `server/` directory kernel.

In [ ]:
import sys
from pathlib import Path

SERVER_ROOT = Path.cwd()
while SERVER_ROOT.name != 'server' and SERVER_ROOT != SERVER_ROOT.parent:
    SERVER_ROOT = SERVER_ROOT.parent
if str(SERVER_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVER_ROOT))

print('SERVER_ROOT:', SERVER_ROOT)

In [ ]:
from db.legacy.etl.load_places import load_places
from db.legacy.etl.build_h3_density import build_h3_density

df = load_places()

In [ ]:
density_df = build_h3_density(
    df[[
        'h3_res10', 'lat', 'lon', 'cuisineType', 'cost', 'venueType',
        'tier', 'tier_d', 'tier_independent',
    ]].rename(columns={
        'h3_res10':    'h3_r10',
        'cuisineType': 'cuisine_type',
        'venueType':   'venue_type',
    }).copy()
)

print(f'Total rows: {len(density_df):,}')
density_df.head()

In [ ]:
print('score_basis values:', sorted(density_df['score_basis'].unique()))
print('score_tier values: ', sorted(density_df['score_tier'].unique()))
print('resolution values: ', sorted(density_df['resolution'].unique()))
print('cuisine_type sample:', sorted(density_df['cuisine_type'].unique())[:6])
print('Duplicate PK rows:  ', density_df.duplicated(
    subset=['tile','resolution','cuisine_type','cost','venue_type','score_basis','score_tier']
).sum())

In [ ]:
OUT_PATH = SERVER_ROOT / 'out' / 'h3_density.csv'
density_df.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')